In [1]:
!git clone https://github.com/zhao-zilong/ssc-cot.git

Cloning into 'ssc-cot'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 134 (delta 54), reused 130 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 348.81 KiB | 1.91 MiB/s, done.
Resolving deltas: 100% (54/54), done.


In [ ]:
!pip install -q peft transformers datasets accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.4 MB/s eta 0:00:00


In [2]:
import os
files = os.listdir("/content/ssc-cot/Dataset")
print(len(files))

100


In [3]:
import os
import json

dataset_path = "/content/ssc-cot/Dataset"

all_data = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.endswith(".json"):
            file_path = os.path.join(root, file)
            with open(file_path, "r", encoding="utf-8") as f:
                try:
                    data = json.load(f)
                    if isinstance(data, list):
                        all_data.extend(data)
                    elif isinstance(data, dict):
                        all_data.append(data)
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

print(f"Total samples loaded: {len(all_data)}")

Total samples loaded: 100


In [4]:
print(all_data[0])
print(all_data[0].keys())

{'question': 'In triangle ABC, the opposite sides of angles A, B, and C are a, b, and c respectively. If tanAtanB=tanAtanC+tanCtanB, then find the value of (a^2 + b^2)/c^2.', 'intermediate-results': [{'result': 'sinAsinB/(cosAcosB)=sinAsinC/(cosAcosC)+sinCsinB/(cosCcosB)', 'step': 1, 'score': 1, 'branch': 'None', 'branch-level': 'None'}, {'result': 'sinAsinBcosC=sinAsinCcosB+sinBsinCcosA', 'step': 2, 'score': 2, 'branch': 'None', 'branch-level': 'None'}, {'result': 'ab*(a^2+b^2-c^2)/(2ab)=ac*(a^2+c^2-b^2)/(2ac)+bc*(b^2+c^2-a^2)/(2bc)', 'step': 3, 'score': 3, 'branch': 'None', 'branch-level': 'None'}, {'result': 'a^2+b^2-c^2=2c^2', 'step': 4, 'score': 4, 'branch': 'None', 'branch-level': 'None'}, {'result': '(a^2+b^2)/c^2=3', 'step': 5, 'score': 5, 'branch': 'None', 'branch-level': 'None'}], 'result': '3'}
dict_keys(['question', 'intermediate-results', 'result'])


In [5]:
from datasets import Dataset

STEP_SEPARATOR = " ки"
POSITIVE_TOKEN = "+"
NEGATIVE_TOKEN = "-"

def convert_prm(sample):
    question = sample.get("question", "")
    intermediate = sample.get("intermediate-results", [])
    steps = [s.get("result", "") for s in intermediate]
    final_result = sample.get("result", None)
    last_step_result = intermediate[-1].get("result", "") if intermediate else ""
    solution_correct = (str(final_result).strip() == str(last_step_result).strip()) if final_result else True
    if solution_correct:
        labels = [POSITIVE_TOKEN] * len(steps)
    else:
        labels = [POSITIVE_TOKEN] * (len(steps) - 1) + [NEGATIVE_TOKEN]
    return {
        "question": question,
        "steps": steps,
        "labels": labels
    }

def format_prm_text(sample):
    question = sample["question"]
    steps = sample["steps"]
    labels = sample["labels"]
    text = question + "\n"
    for step, label in zip(steps, labels):
        text += step + STEP_SEPARATOR + label + "\n"
    return {"text": text}

hf_data = [convert_prm(x) for x in all_data if isinstance(x, dict) and x.get("intermediate-results")]
dataset = Dataset.from_list(hf_data)
dataset = dataset.map(format_prm_text)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)
print(dataset["train"][0]["text"][:500])


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'steps', 'labels', 'text'],
        num_rows: 90
    })
    test: Dataset({
        features: ['question', 'steps', 'labels', 'text'],
        num_rows: 10
    })
})
In triangle ABC, the angles A, B, and C correspond to the three sides a, b, and c. If 9a^2 + 9b^2 - 19c^2 = 0, then find the value of cotC/(cotA + cotB)
cotC/(cotA + cotB) = (cosC/sinC)/(cosA/sinA + cosB/sinB) ки+
(cosC/sinC)/(cosA/sinA + cosB/sinB) = (cosC/sinC)/((cosAsinB+cosBsinA)/(sinAsinB)) ки+
(cosC/sinC)/((cosAsinB+cosBsinA)/(sinAsinB)) = sinAsinB/((sinC)^2)cosC ки+
sinAsinB/((sinC)^2)cosC = ab/c^2*(a^2+b^2-c^2)/(2ab) ки+
ab/c^2*(a^2+b^2-c^2)/(2ab) = (a^2+b^2-c^2)/(2c^2) ки+
(a^2+b^2-c^2)


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [10]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [11]:
MAX_LENGTH = 512

def tokenize(sample):
    result = tokenizer(
        sample["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_train = dataset["train"].map(tokenize, batched=True, remove_columns=dataset["train"].column_names)
tokenized_eval = dataset["test"].map(tokenize, batched=True, remove_columns=dataset["test"].column_names)

print(tokenized_train)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 90
})


In [13]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

import torch

# Important
torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="./prm_lora_output",

    # smaller batch
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    # simulate larger batch
    gradient_accumulation_steps=8,

    num_train_epochs=3,

    learning_rate=2e-4,
    warmup_steps=20,

    # use bf16 if supported
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    logging_steps=10,

    # reduce eval overhead
    eval_strategy="steps",
    eval_steps=500,

    save_strategy="steps",
    save_steps=500,

    load_best_model_at_end=True,

    # memory optimizations
    gradient_checkpointing=True,
    dataloader_pin_memory=False,

    # huge VRAM saver
    optim="paged_adamw_8bit",

    report_to="none",

    # prevents extra memory spikes
    remove_unused_columns=False
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

model.config.use_cache = False

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.


ImportError: You need to install `bitsandbytes` in order to use bitsandbytes optimizers: `pip install -U bitsandbytes`

In [ ]:
model.save_pretrained("./prm_lora_final")
tokenizer.save_pretrained("./prm_lora_final")
print("Model saved to ./prm_lora_final")

Model saved to ./prm_lora_final


In [ ]:
model_path = "/content/prm_lora_final"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

In [ ]:
prompt = """You are a math solution grader.

DO NOT solve the problem again.
ONLY evaluate the given steps.

Scoring rule (VERY IMPORTANT):
- Score each step from 0–10 based on BOTH correctness AND importance.
- Importance = how critical the step is to reaching the final answer.

Guidelines:
- Minor algebra / identity expansion → low importance (1–3)
- Intermediate simplifications → medium importance (4–6)
- Key transformation / insight → high importance (7–9)
- Final conclusion / decisive step → highest importance (10)

If a step is correct but trivial → give LOW score.
If a step is critical for solving → give HIGH score.

Question:
sinA - cosB = 3cosA - 3sinB and sin(A+B) ≠ 1, find sin(A-B)

Solution to grade:
1. sinA - cosB = 3cosA - 3sinB
2. ⇒ sinA - 3cosA = 3sinB - cosB
3. ⇒ sinA - 3cosA = √10 sin(A - φ), tanφ = 3
4. ⇒ 3sinB - cosB = √10 sin(B - φ)
5. ⇒ sin(A - φ) = sin(B - φ)
6. ⇒ A = B or A + B = π
7. ⇒ sin(A - B) = 0

Task:
For EACH step output:
- correctness: correct / incorrect
- importance: low / medium / high
- explanation: one short line
- score: (0–10)

Then output FINAL SCORE based on overall reasoning quality.

Output format (STRICT):

Step 1:
correctness:
importance:
explanation:
score:

Step 2:
...

Score formula:
score = importance_weight × correctness

Where:
low = 2
medium = 5
high = 8–10
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.1,        # deterministic
    top_p=1.0,              # no nucleus sampling restriction needed
    do_sample=False,        # disable sampling
    repetition_penalty=1.0, # avoid distortion
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

In [ ]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

You are a math solution grader.

DO NOT solve the problem again.
ONLY evaluate the given steps.

Scoring rule (VERY IMPORTANT):
- Score each step from 0–10 based on BOTH correctness AND importance.
- Importance = how critical the step is to reaching the final answer.

Guidelines:
- Minor algebra / identity expansion → low importance (1–3)
- Intermediate simplifications → medium importance (4–6)
- Key transformation / insight → high importance (7–9)
- Final conclusion / decisive step → highest importance (10)

If a step is correct but trivial → give LOW score.
If a step is critical for solving → give HIGH score.

Question:
sinA - cosB = 3cosA - 3sinB and sin(A+B) ≠ 1, find sin(A-B)

Solution to grade:
1. sinA - cosB = 3cosA - 3sinB  
2. ⇒ sinA - 3cosA = 3sinB - cosB  
3. ⇒ sinA - 3cosA = √10 sin(A - φ), tanφ = 3  
4. ⇒ 3sinB - cosB = √10 sin(B - φ)  
5. ⇒ sin(A - φ) = sin(B - φ)  
6. ⇒ A = B or A + B = π  
7. ⇒ sin(A - B) = 0  

Task:
For EACH step output:
- correctness: